# Training notebook (Kaggle)
**Settings (right-hand panel):** Accelerator = **GPU T4 x2**, Internet = **On**. **Input:** the `dam-ecl` code dataset.

Uses (set `PLANS` in cell 2):
- `PLANS = "round4"` (**current**, about 1 hour with 2 accounts): seeds 3-4 for the duplicates and step-imbalance
  experiments, plus shuffled gaps in the duplicates setting (29 runs).
- `PLANS = "round3"` (done, about 2.5 hours with 2 accounts): duplicated classes, step imbalance with 2 or 5
  minority classes, MNIST/Fashion-MNIST with content-mass labels, and DAM-Mix with power 1 on every dataset (68 runs).
  Same account settings as below (`ACCOUNT = 0` / `1`, `NUM_ACCOUNTS = 2`).
- `PLANS = "paper,controls"` (**for the journal paper**, about 3 hours with 2 accounts): Fashion-MNIST-LT, SVHN-LT and
  MNIST-LT (7 methods x 3 seeds), a third CIFAR-100-LT seed, the DAM-Mix weights, and the CIFAR-10 controls
  (shuffled gap, gap power 1/3, probe images for the baselines). Run it on BOTH accounts at the same time:
  account 1 with `ACCOUNT = 0`, account 2 with `ACCOUNT = 1` (`NUM_ACCOUNTS = 2` on both).
- `PLANS = "final"` (default, about 10 minutes): trains the final **DAM-Mix** model once and saves its weights to
  `final_model/` (the zip contains the weights, results and embeddings).
- `PLANS = "main,ablation,fair,ir,c100,sens,v3,v4"`: reproduces every experiment in the report (about 12 GPU-hours;
  split it across accounts with `NUM_ACCOUNTS`).

Then **Save Version → Save & Run All (Commit)**. Download the `results_..._accountX.zip` from the **Output** tab.

In [ ]:
# 1) copy the code into the working directory + check GPU and Internet
import glob, os, shutil
src = os.path.dirname(glob.glob("/kaggle/input/**/run_all.py", recursive=True)[0])
if not os.path.exists("/kaggle/working/dam-ecl"):
    shutil.copytree(src, "/kaggle/working/dam-ecl")
os.chdir("/kaggle/working/dam-ecl")
!nvidia-smi -L
!curl -sI https://huggingface.co | head -1   # must print HTTP/2 200; otherwise turn Internet ON

In [ ]:
# 2) settings
ACCOUNT = 0               # 0 on the first account, 1 on the second
NUM_ACCOUNTS = 2          # 2 Kaggle accounts in parallel
EPOCHS = 100
PLANS = "round4"           # round 4: extra seeds for A/B + shuffled gaps with duplicates

In [ ]:
# 3) sanity tests (about 1 minute)
!python -m pytest -q tests

In [ ]:
# 4) train (see progress_part*.txt for live progress) and zip the results (+ final_model/ weights)
!ACCOUNT={ACCOUNT} NUM_ACCOUNTS={NUM_ACCOUNTS} EPOCHS={EPOCHS} PLANS={PLANS} bash kaggle_run.sh
!cp results_*_account{ACCOUNT}.zip /kaggle/working/ 2>/dev/null; ls -la /kaggle/working